In [ ]:
"""
Solving a System of Equations using Python (NumPy/SymPy)
Equivalent to MATLAB Symbolic Math Toolbox using 'solve' and 'linsolve'

ODE: y'' = x + y
Boundary Conditions: y(0) = y(1) = 0
Step size: h = 1/4
"""

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

# ── Symbolic solve (equivalent to MATLAB 'solve') ──────────────────────────

print("Solving the equations using sympy solve command")

y1, y2, y3 = sp.symbols('y1 y2 y3')

eqn1 = sp.Eq(-33*y1 + 16*y2,        sp.Rational(1, 4))
eqn2 = sp.Eq( 16*y1 - 33*y2 + 16*y3, sp.Rational(1, 2))
eqn3 = sp.Eq( 16*y2 - 33*y3,        sp.Rational(3, 4))

sol = sp.solve([eqn1, eqn2, eqn3], [y1, y2, y3])

y1sol = sol[y1]
y2sol = sol[y2]
y3sol = sol[y3]

print(f"y1sol = {y1sol}")
print(f"y2sol = {y2sol}")
print(f"y3sol = {y3sol}")

print(f"\ny1 = {float(y1sol):.4f}")
print(f"y2 = {float(y2sol):.4f}")
print(f"y3 = {float(y3sol):.4f}")

# ── Matrix/linsolve (equivalent to MATLAB 'linsolve') ──────────────────────

print("\nSolving the equations using numpy linsolve (np.linalg.solve)")

A, B = sp.linear_eq_to_matrix([eqn1, eqn2, eqn3], [y1, y2, y3])
print(f"A =\n{A}")
print(f"B =\n{B}")

A_np = np.array(A.tolist(), dtype=float)
B_np = np.array(B.tolist(), dtype=float).flatten()
Y = np.linalg.solve(A_np, B_np)

print(f"\ny1 = {Y[0]:.4f}")
print(f"y2 = {Y[1]:.4f}")
print(f"y3 = {Y[2]:.4f}")

# ── Exact solution ─────────────────────────────────────────────────────────

exact_solution = lambda x: (np.sinh(x) / np.sinh(1)) - x

y1_exact = exact_solution(0.25)
y2_exact = exact_solution(0.50)
y3_exact = exact_solution(0.75)

print(f"\ny1 (exact) = {y1_exact:.4f}")
print(f"y2 (exact) = {y2_exact:.4f}")
print(f"y3 (exact) = {y3_exact:.4f}")

# ── Comparison table: solve vs linsolve ───────────────────────────────────

print("\n\n")
print("+----------+---------------+-----------------+-------------------+")
print("| Variable | solve Method  | linsolve Method |    Difference     |")
print("+----------+---------------+-----------------+-------------------+")
for label, sym_val, num_val in [("y1", y1sol, Y[0]),
                                  ("y2", y2sol, Y[1]),
                                  ("y3", y3sol, Y[2])]:
    diff = abs(float(sym_val) - num_val)
    diff = 0.0 if diff < 1e-10 else diff 
    print(f"|   {label}     |   {float(sym_val):11.6f} |   {num_val:11.6f}   |   {diff:13.2e} |")
    print("+----------+---------------+-----------------+-------------------+")

# ── Plot ───────────────────────────────────────────────────────────────────

print("\nGenerating plot...")

x_numerical = np.array([0.25, 0.50, 0.75])
y_numerical  = np.array([float(y1sol), float(y2sol), float(y3sol)])

x_exact = np.linspace(0, 1, 100)
y_exact = exact_solution(x_exact)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_numerical, y_numerical, 'ro-', linewidth=2, markersize=10,
        markerfacecolor='r', label='Numerical Solution (solve method)')
ax.plot(x_exact, y_exact, 'b-', linewidth=2, label='Exact Solution')

ax.set_xlabel('x', fontsize=12, fontweight='bold')
ax.set_ylabel('y(x)', fontsize=12, fontweight='bold')
ax.set_title("Comparison of Numerical and Exact Solutions for y'' = x + y",
             fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.set_xlim(0, 1)
ax.set_ylim(-0.12, 0)

ax.text(0.05, -0.107, 'Boundary conditions: y(0) = 0, y(1) = 0', fontsize=10)
ax.text(0.05, -0.114, 'Step size h = 1/4', fontsize=10)

for xi, yi in zip(x_numerical, y_numerical):
    ax.annotate(f'  ({xi:.2f}, {yi:.4f})', xy=(xi, yi), fontsize=9,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          edgecolor='gray', alpha=0.8))

plt.tight_layout()

# Save plot with error handling
try:
    # Try to save to the specified directory (create if needed)
    output_dir = 'Outputs'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'ode_solution_plot.png')
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"Plot saved to: {output_path}")
except (PermissionError, OSError) as e:
    # If can't write to that directory, save to current directory
    output_path = 'ode_solution_plot.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"Plot saved to current directory: {output_path}")

plt.show()
print("Plot generated successfully!")

# ── Error analysis ─────────────────────────────────────────────────────────

print("\nError Analysis:")
print("+----------+---------------+---------------+-------------------+")
print("|   x      | Numerical y   | Exact y       | Absolute Error    |")
print("+----------+---------------+---------------+-------------------+")
for xi, yi in zip(x_numerical, y_numerical):
    abs_err = abs(yi - exact_solution(xi))
    print(f"|  {xi:.2f}    |   {yi:11.6f} |   {exact_solution(xi):11.6f} |   {abs_err:13.2e} |")
print("+----------+---------------+---------------+-------------------+")

print("\nDetailed Error Analysis:")
print("+----------+---------------+---------------+-------------------+------------------+")
print("|   x      | Numerical y   | Exact y       | Absolute Error    | Relative Error   |")
print("+----------+---------------+---------------+-------------------+------------------+")
for xi, yi in zip(x_numerical, y_numerical):
    abs_err = abs(yi - exact_solution(xi))
    rel_err = abs_err / abs(exact_solution(xi)) * 100
    print(f"|  {xi:.2f}    |   {yi:11.6f} |   {exact_solution(xi):11.6f} |   {abs_err:13.2e} |      {rel_err:6.2f}%    |")
print("+----------+---------------+---------------+-------------------+------------------+")

# ── Convergence metrics ────────────────────────────────────────────────────

exact_at_nodes = exact_solution(x_numerical)
max_abs_err  = np.max(np.abs(y_numerical - exact_at_nodes))
rmse         = np.sqrt(np.mean((y_numerical - exact_at_nodes)**2))
avg_rel_err  = np.mean(np.abs((y_numerical - exact_at_nodes) / exact_at_nodes)) * 100

print("\nConvergence Metrics:")
print(f"Maximum Absolute Error:        {max_abs_err:.2e}")
print(f"Root Mean Square Error (RMSE): {rmse:.2e}")
print(f"Average Relative Error:        {avg_rel_err:.2f}%")